# What is an attractive merchant for Snap?

Notebook 01 explored the quality of the data. This notebook uses the `snap` data set, which is internal merchant list. The list can help understand what are common in for active, successful merchants.

**Flow:** **02 what makes an attractive merchant** → 03 where to find more → 04 who Sales should contact first → 05 how we measure it

## Setup

In [36]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import plotly.express as px
import kaleido ##needed if saving plotly figures as images

# Find the repo root by walking up until the warehouse folder is found,
# so the notebook works whatever folder the kernel starts in.
here = Path.cwd().resolve()
repo_root = next(p for p in [here, *here.parents] if (p / "warehouse" / "snap-assessment.duckdb").exists())

# read_only=True so the notebook never blocks `dbt build` from writing to the file.
con = duckdb.connect(str(repo_root / "warehouse" / "snap-assessment.duckdb"), read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a Pandas DataFrame."""
    return con.sql(sql).df()

## snap

Let's look at the internal list to see if we can find some similarities of accounts that have been successful for Snap.

Reminder: using the `stg_snap` table, that clean & normalized fields and rank to remove dupe records (under [Assumptions](../docs/assumptions.md)).

**product type**

In [37]:
q("""

WITH merchant_segments AS (
    SELECT
        product_type,
        COUNT(DISTINCT mid) AS active_merchants,
        count(distinct industry) as n_industries,
        SUM(apps_ytd) AS apps_ytd,
        SUM(aprv_ytd) AS aprv_ytd,
        SUM(cmpl_ytd) AS cmpl_ytd,
        SUM(fund_ytd) AS fund_ytd
    FROM "snap-assessment".staging.stg_snap
    WHERE NOT is_inactivated
      AND dupe_record_rank = 1
    GROUP BY ALL
)

SELECT
    product_type,

    -- Merchant coverage
    active_merchants,
    n_industries,

    -- Application volume
    apps_ytd,

    -- Conversion
    ROUND(100.0 * aprv_ytd / NULLIF(apps_ytd, 0), 2) AS approval_rate,
    ROUND(100.0 * cmpl_ytd / NULLIF(apps_ytd, 0), 2) AS application_completion_rate,
    ROUND(100.0 * cmpl_ytd / NULLIF(aprv_ytd, 0), 2) AS approved_completion_rate,

    -- Funding
    fund_ytd as total_funding_ytd,
    ROUND(fund_ytd / NULLIF(active_merchants, 0), 2) AS avg_funding_per_merchant

FROM merchant_segments
ORDER BY product_type, fund_ytd DESC

""")

,product_type,active_merchants,n_industries,apps_ytd,approval_rate,application_completion_rate,approved_completion_rate,total_funding_ytd,avg_funding_per_merchant
0,Lease,425,11,34893.0,60.28,39.92,66.22,20632661.0,48547.44
1,Loan,229,6,6452.0,51.49,34.61,67.22,3534148.0,15432.96


**product and industry**

In [38]:
df_snap_performance = q("""

SELECT
    product_type,
    industry,
    COUNT(DISTINCT mid) AS active_merchants,
    SUM(apps_ytd) AS apps_ytd,
    SUM(aprv_ytd) AS aprv_ytd,

    ROUND(
        100.0 * SUM(aprv_ytd) / NULLIF(SUM(apps_ytd), 0), 2
    ) AS approval_rate,

    SUM(cmpl_ytd) AS cmpl_ytd,

    ROUND(
        100.0 * SUM(cmpl_ytd) / NULLIF(SUM(apps_ytd), 0), 2
    ) AS completion_rate,

    SUM(fund_ytd) AS fund_ytd

FROM "snap-assessment".staging.stg_snap
WHERE NOT is_inactivated
  AND dupe_record_rank = 1
GROUP BY ALL
HAVING SUM(COALESCE(fund_ytd, 0)) > 0
ORDER BY apps_ytd DESC;

""")

In [39]:
fig = px.scatter(
    df_snap_performance,
    x="apps_ytd",
    y="approval_rate",
    size="fund_ytd",  ##size of the bubble is the total funding YTD
    color="product_type",  ##color of the bubble is the product type
    hover_name="industry", ##hover over the bubble to see the industry name
    hover_data={
        "active_merchants": ":,",
        "apps_ytd": ":,",
        "aprv_ytd": ":,",
        "approval_rate": ":.2f",
        "cmpl_ytd": ":,",
        "completion_rate": ":.2f",
        "fund_ytd": ":$,.0f"
    },
    size_max=45,
    title="Snap Merchant Performance by Industry and Product",
    labels={
        "apps_ytd": "Applications YTD",
        "approval_rate": "Approval Rate (%)",
        "product_type": "Product Type",
        "fund_ytd": "Funding YTD"
    }
)

fig.update_layout(
    template="plotly_white",
    height=650,
    legend_title_text="Product Type"
)

fig.update_yaxes(ticksuffix="%")

fig.show()

In case the image doesn't show while browsing the repo:

![Snap Merchant Performance](figures/snap_industry_performance.png)

created using `fig.write_image("figures/snap_industry_performance.png")`

**How I'd interpret the plot above**

- High volume / high approval
    - Strong market
    - Established segments with demonstrated application demand and conversion.

- Low volume / high approval
    - Explore growth
    - Potentially underserved segments worth investigating.

- High volume / low approval
    - Investigate
    - Demand exists, but approval performance warrants further investigation.

- Low volume / low approval
    - Lower priority
    - Less demonstrated demand and weaker conversion.

One caveat: low approval rates don't necessarily mean an industry is unattractive. They could reflect customer credit profiles, application quality, or differences in Snap's underwriting criteria. I'd avoid treating approval rate as a direct measure of merchant quality.

In [40]:
df_snap_performance

,product_type,industry,active_merchants,apps_ytd,aprv_ytd,approval_rate,cmpl_ytd,completion_rate,fund_ytd
0,Lease,FURNITURE,98,9731.0,5809.0,59.70,3753.0,38.57,5476968.0
1,Lease,TIRES_RIMS,159,9395.0,6432.0,68.46,4692.0,49.94,6660048.0
2,Loan,AUTO_SERVICE,211,5758.0,3002.0,52.14,2025.0,35.17,3246507.0
3,Lease,JEWELRY,36,4228.0,2156.0,50.99,1374.0,32.50,2474761.0
4,Lease,OTHER,11,4067.0,1899.0,46.69,996.0,24.49,1638829.0
5,Lease,AUTO_SERVICE,39,2380.0,1718.0,72.18,1206.0,50.67,1826549.0
6,Lease,APPLIANCES,24,1682.0,1113.0,66.17,745.0,44.29,1021383.0
7,Lease,CAR_AUDIO,36,1471.0,937.0,63.70,607.0,41.26,916177.0
8,Lease,MUSICAL_INSTRUMENTS,1,939.0,444.0,47.28,209.0,22.26,188545.0
9,Lease,MATTRESSES,15,740.0,380.0,51.35,290.0,39.19,330061.0


This helps provide what industry types do the best business so far.

I think some work could also be done to explore if location is a factor, but with the addresses all over the place. I'm going to place this on hold.

In [41]:
df_snap_performance.describe(include="all")

,product_type,industry,active_merchants,apps_ytd,aprv_ytd,approval_rate,cmpl_ytd,completion_rate,fund_ytd
count,16,16,16.000000,16.000000,16.00000,16.000000,16.000000,16.000000,1.600000e+01
unique,2,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Lease,FURNITURE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,11,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,40.812500,2583.875000,1522.12500,57.040000,1010.062500,35.651875,1.510426e+06
std,NaN,NaN,62.149517,3225.185222,2012.74343,9.419722,1397.074299,11.350508,2.042871e+06
min,NaN,NaN,1.000000,57.000000,38.00000,37.060000,11.000000,19.040000,1.603200e+04
25%,NaN,NaN,3.500000,190.750000,101.75000,51.260000,63.250000,24.032500,8.273475e+04
50%,NaN,NaN,13.000000,1205.000000,690.50000,57.770000,448.500000,36.870000,6.231190e+05
75%,NaN,NaN,36.750000,4107.250000,1963.25000,64.317500,1248.000000,44.420000,1.988602e+06


## Takeaways

**1. Leasing drives Snap's results.**
Lease merchants (425) produced `$20.6M` of `$24.2M` funded YTD, about 85%. The average lease merchant funded about `$48.5K`, against about `$15.4K` for a loan merchant, roughly 3x. Leases also have a higher approval rate (60.3% vs 51.5%).

**2. Three industries generate most of the funding.**
Tires, furniture and auto service together account for about 72% of funded dollars, and jewelry adds another 10%.

| Industry | Funded YTD | Share |
|---|---|---|
| Tires & rims | $6.7M | 28% |
| Furniture | $5.5M | 23% |
| Auto service | $5.1M | 21% |
| Jewelry | $2.5M | 10% |

**3. Volume alone doesn't mean a merchant is attractive.**
- Lease jewelry has 36 merchants and funded about $69K each, the highest per merchant.
- Lease auto service has the best approval rate of the large segments (72%) and funded about $47K per merchant.
- Loan auto service has the most merchants (211) but only about `$15K` funded each. Many merchants there are small or barely active.
- Collision repair (loan) is the weakest large-ish segment: 37% approval and 19% completion.

**4. Approval isn't the main leak. Completion is.**
Only 35% to 40% of applications end up completed, even though 66% to 67% of approved applications complete. Merchants who help customers finish the application are worth more than ones who just generate applications.

**5. What this means for the rest of the project.**
I treat "attractive" as a merchant in tires, furniture, auto service, jewelry or appliances. That is what notebook 03 searches for in the other lists.

**Limits**
- These numbers are YTD and cover active merchants only. They show where Snap has succeeded, not where it could.
- Approval rate reflects customer credit and underwriting as well as merchant quality.
- Small segments (for example 1 to 5 merchants) are anecdotes, not patterns.
- I did not test location. The addresses are too inconsistent to do it reliably here, so it is a next step.

## close connection

In [42]:
con.close()